# Phase 6.2: Model Training (EfficientNet-B4)
Bu notebook'ta, ViT modeli ile karşılaştırmak üzere (A/B Testing) EfficientNet-B4 CNN mimarisini eğiteceğiz.

## İçerik:
- Modelin Yüklenmesi (timm kütüphanesi ile EfficientNet-B4)
- Classifier Head'in Değiştirilmesi (102 Sınıf için)
- Optimizer (AdamW), Loss Function (CrossEntropy) ve Scheduler
- Eğitim ve Validasyon Döngüsü (Training Loop) AMP destekli
- Metrikler (Accuracy, F1-Score) ve Early Stopping

In [1]:
import os
import time
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from sklearn.metrics import f1_score, accuracy_score
import timm

# Dataset modülünden fonksiyonları alıyoruz
import sys
sys.path.append('../') 
from dataset import get_dataloaders

c:\Users\Asus\OneDrive\Masaüstü\capstone_final\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Config
DATA_DIR = '../data/classification'
BATCH_SIZE = 16
IMG_SIZE = 224 # ViT ile adil kıyaslama için aynı boyutu tutuyoruz
NUM_EPOCHS = 30
LEARNING_RATE = 3e-4 # CNN'ler genellikle ViT'lere göre biraz daha yüksek LR sever
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Kullanılan Cihaz: {DEVICE}")

Kullanılan Cihaz: cuda


## 1. Dataloader'ları Başlatma

In [3]:
train_loader, val_loader, test_loader, num_classes = get_dataloaders(
    data_dir=DATA_DIR, 
    img_size=IMG_SIZE, 
    batch_size=BATCH_SIZE
)
print(f"Toplam Sınıf Sayısı: {num_classes}")

c:\Users\Asus\OneDrive\Masaüstü\capstone_final\venv\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Toplam Sınıf Sayısı: 102


## 2. EfficientNet-B4 Modelinin Yüklenmesi
Pre-trained ImageNet ağırlıkları ile EfficientNet-B4 modelini kuruyoruz.

In [4]:
# Proposal'da belirtilen EfficientNet-B4 modeli.
model = timm.create_model('efficientnet_b4', pretrained=True, num_classes=num_classes)
model = model.to(DEVICE)
print("EfficientNet-B4 modeli başarıyla yüklendi.")

c:\Users\Asus\OneDrive\Masaüstü\capstone_final\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asus\.cache\huggingface\hub\models--timm--efficientnet_b4.ra2_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


EfficientNet-B4 modeli başarıyla yüklendi.


## 3. Loss, Optimizer ve Scheduler

In [5]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

## 4. Eğitim Döngüsü (Training Loop) & Early Stopping
Bu döngüde GPU bellek kullanımını azaltmak için PyTorch AMP (Automatic Mixed Precision) eklenmiştir.

In [6]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=25, patience=5):
    since = time.time()
    
    best_model_wts = copy.deepcopy(model.state_dict())
    best_f1 = 0.0
    epochs_no_improve = 0
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}

    scaler = torch.amp.GradScaler('cuda')

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                dataloader = train_loader
            else:
                model.eval()
                dataloader = val_loader

            running_loss = 0.0
            all_preds = []
            all_labels = []

            for inputs, labels in tqdm(dataloader, desc=f"{phase.capitalize()} Batch"):
                inputs = inputs.to(DEVICE)
                labels = labels.to(DEVICE)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    with torch.amp.autocast('cuda', enabled=(phase == 'train')):
                        outputs = model(inputs)
                        loss = criterion(outputs, labels)
                    
                    _, preds = torch.max(outputs, 1)

                    if phase == 'train':
                        scaler.scale(loss).backward()
                        scaler.step(optimizer)
                        scaler.update()

                running_loss += loss.item() * inputs.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
            
            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = accuracy_score(all_labels, all_preds)
            epoch_f1 = f1_score(all_labels, all_preds, average='macro')

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} Macro F1: {epoch_f1:.4f}')
            
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc)
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc)
                history['val_f1'].append(epoch_f1)

                if epoch_f1 > best_f1:
                    best_f1 = epoch_f1
                    best_model_wts = copy.deepcopy(model.state_dict())
                    epochs_no_improve = 0
                    torch.save(model.state_dict(), 'best_efficientnet_model.pth')
                else:
                    epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f'Early stopping triggered after {patience} epochs without F1 improvement.')
            break

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Macro F1: {best_f1:4f}')

    model.load_state_dict(best_model_wts)
    return model, history

In [7]:
# Eğitimi Başlat
best_model, history = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs=NUM_EPOCHS, patience=5)

Epoch 1/30
----------


Train Batch: 100%|██████████| 2819/2819 [10:47<00:00,  4.36it/s]


Train Loss: 2.3026 Acc: 0.5731 Macro F1: 0.5676


Val Batch: 100%|██████████| 470/470 [00:58<00:00,  8.08it/s]


Val Loss: 2.2565 Acc: 0.5578 Macro F1: 0.5147

Epoch 2/30
----------


...

Val Loss: 1.9136 Acc: 0.7187 Macro F1: 0.6562

Epoch 30/30
----------


Train Batch: 100%|██████████| 2819/2819 [10:55<00:00,  4.30it/s]


Train Loss: 0.8169 Acc: 0.9950 Macro F1: 0.9950


Val Batch: 100%|██████████| 470/470 [00:55<00:00,  8.45it/s]

Val Loss: 1.9122 Acc: 0.7198 Macro F1: 0.6586

Training complete in 356m 31s
Best val Macro F1: 0.659910
